# 03. Baseline Real Estate Valuation Models (Exp 1 vs Exp 2)
**Project:** *Does Bangalore Traffic Explain House Prices? Uncertainty-Aware Price Prediction with Road-Network GNNs and Bayesian Heads*  
**Research Question 1 (Predictive):** *Do peak-hour commute features improve price prediction over non-spatial tabular baselines?*

---
### Objectives:
1. **Experiment 1**: Train non-spatial tabular baselines (XGBoost & MLP) using structural property features only.
2. **Experiment 2**: Train commute-aware tabular models incorporating directional commute times to 5 tech hubs, metro distance, and congestion ratios.
3. Quantify the predictive gain of traffic features under:
   - **Random Split** (Standard reference)
   - **Leave-Locality-Out (LLO 5-Fold)** (Zero spatial leakage)
4. Analyze feature importance to understand which tech corridors drive house valuations.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBRegressor

from src.logger import ServerLogger
from src.evaluation import compute_point_metrics, log_evaluation_summary

logger = ServerLogger.get_logger()
logger.info("Notebook 03: Baseline Models Exp 1 & 2 initialized.")

## 1. Load Prepared Dataset and Feature Definitions

In [ ]:
df = pd.read_csv('../data/bangalore_housing_spatial_commute_dataset.csv')

# Tabular listing attributes (Exp 1)
tabular_features = ['bhk', 'total_sqft_cleaned', 'bath_cleaned', 'balcony_cleaned', 'is_ready_to_move']
area_dummies = pd.get_dummies(df['area_type'], prefix='area', drop_first=True)
df = pd.concat([df, area_dummies], axis=1)
tabular_features.extend(list(area_dummies.columns))

# Explicit Commute & Transit Features (Exp 2)
commute_features = [
    'commute_whitefield_peak_min', 'commute_whitefield_offpeak_min',
    'commute_bellandur_orr_peak_min', 'commute_bellandur_orr_offpeak_min',
    'commute_electronic_city_peak_min', 'commute_electronic_city_offpeak_min',
    'commute_manyata_tech_park_peak_min', 'commute_manyata_tech_park_offpeak_min',
    'commute_mg_road_cbd_peak_min', 'commute_mg_road_cbd_offpeak_min',
    'min_commute_peak_min', 'avg_commute_peak_min',
    'avg_congestion_ratio', 'dist_to_metro_km', 'commute_metro_access_min'
]

print(f"Exp 1 Tabular Features ({len(tabular_features)}): {tabular_features}")
print(f"Exp 2 Commute Features ({len(commute_features)}): {commute_features}")

## 2. Train Exp 1 vs. Exp 2 on Random Split Benchmark

In [ ]:
train_mask = (df['split_random'] == 'train').values
test_mask = (df['split_random'] == 'test').values

# Feature matrices
X_tab = df[tabular_features].values
X_comm = df[commute_features].values
X_all = np.concatenate([X_tab, X_comm], axis=1)
y = df['log_price'].values

# Fit Exp 1 (Tabular Only)
xgb_exp1 = XGBRegressor(n_estimators=150, max_depth=6, learning_rate=0.08, random_state=42)
xgb_exp1.fit(X_tab[train_mask], y[train_mask])
pred_exp1 = xgb_exp1.predict(X_tab[test_mask])
m_exp1 = compute_point_metrics(y[test_mask], pred_exp1)

# Fit Exp 2 (Tabular + Commute)
xgb_exp2 = XGBRegressor(n_estimators=150, max_depth=6, learning_rate=0.08, random_state=42)
xgb_exp2.fit(X_all[train_mask], y[train_mask])
pred_exp2 = xgb_exp2.predict(X_all[test_mask])
m_exp2 = compute_point_metrics(y[test_mask], pred_exp2)

df_comp = pd.DataFrame([
    {"Model": "Exp 1: XGBoost (Tabular Only)", **m_exp1},
    {"Model": "Exp 2: XGBoost (Tabular + Commute)", **m_exp2}
])
df_comp

## 3. Feature Importance Analysis (Which Corridors Matter Most?)

In [ ]:
all_feature_names = tabular_features + commute_features
importances = xgb_exp2.feature_importances_
feat_df = pd.DataFrame({"Feature": all_feature_names, "Importance": importances})
feat_df = feat_df.sort_values(by="Importance", ascending=False).head(12)

plt.figure(figsize=(10, 5))
sns.barplot(x="Importance", y="Feature", data=feat_df, palette="viridis")
plt.title("Top Feature Importances in Commute-Aware Real Estate Valuation", fontsize=12)
plt.tight_layout()
plt.show()

## 4. Leave-Locality-Out (LLO 5-Fold) Generalization Gap

In [ ]:
llo_exp1_scores = []
llo_exp2_scores = []

for fold in range(5):
    test_fold = (df['split_llo_fold'] == fold).values
    train_fold = (~test_fold)
    
    # Train Exp 1
    m1 = XGBRegressor(n_estimators=100, max_depth=6, learning_rate=0.08, random_state=42)
    m1.fit(X_tab[train_fold], y[train_fold])
    p1 = m1.predict(X_tab[test_fold])
    llo_exp1_scores.append(compute_point_metrics(y[test_fold], p1)['rmse'])
    
    # Train Exp 2
    m2 = XGBRegressor(n_estimators=100, max_depth=6, learning_rate=0.08, random_state=42)
    m2.fit(X_all[train_fold], y[train_fold])
    p2 = m2.predict(X_all[test_fold])
    llo_exp2_scores.append(compute_point_metrics(y[test_fold], p2)['rmse'])

print(f"5-Fold LLO Mean RMSE (Exp 1 Tabular Only):    {np.mean(llo_exp1_scores):.4f} +/- {np.std(llo_exp1_scores):.4f}")
print(f"5-Fold LLO Mean RMSE (Exp 2 Tabular+Commute): {np.mean(llo_exp2_scores):.4f} +/- {np.std(llo_exp2_scores):.4f}")
logger.info("Notebook 03 execution completed.")